# Avito: кандидатогенерация для поиска услуг

Для каждого запроса из `benchmark_queries.parquet` нужно вернуть до 50 `item_id` из корпуса (189 212 объявлений). Метрика — **Recall@50**.

**Подход.** Для каждого запроса собирается пул из ~1000 кандидатов несколькими генераторами (BM25 по полям объявления, априорная вероятность микрокатегории, близость по локации), затем 50 лучших выбираются линейной формулой над признаками пары «запрос — объявление». Веса формулы подбираются на локальной валидации, которая повторяет состав бенчмарка.

| Раздел | Что происходит |
|---|---|
| 0. Окружение | код из репозитория, зависимости, сиды |
| 1. Данные | загрузка и приведение типов |
| 2. EDA | ответы на вопросы, от которых зависят решения |
| 3. Валидация | псевдо-бенчмарк из train, стратифицированный под бенчмарк |
| 4. Пул кандидатов | генераторы и их полнота на валидации |
| 5. Веса формулы | подбор и честная оценка |
| 6. Анализ качества | метрика по сегментам, примеры промахов |
| 7. Бенчмарк | предсказание, `answer.csv`, проверка формата |
| 8. Артефакты | отчёт, md5, пулы для следующих этапов |

**Запуск.** Kaggle: подключить датасет, включить Internet, для приватного репозитория — секрет `GITHUB_TOKEN`; GPU не нужен. Локально: `pip install -r requirements.txt`, данные в `./data` (или `DATA_DIR`). Подробнее — в README.

**Open-source:** numpy, pandas, scipy, scikit-learn (`CountVectorizer`), pyarrow, pymorphy3. Внешних API нет.

## 0. Окружение

Ячейка находит код решения: локально — папку репозитория рядом с ноутбуком; на Kaggle — клонирует репозиторий (токен берётся из Kaggle Secrets и не попадает ни в вывод, ни на диск). Число потоков BLAS фиксируется **до** импорта numpy.

In [1]:
import base64, os, subprocess, sys
from pathlib import Path

N_THREADS = 4
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = str(N_THREADS)

IS_KAGGLE = Path("/kaggle/input").exists()
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"          # для финального прогона — хеш коммита
REPO_DIR = Path("/kaggle/working/avito-candgen")


def github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def checkout_repo() -> Path:
    """Клонирует (или обновляет) репозиторий и переключается на REPO_REF."""
    token = github_token()
    if not REPO_DIR.exists():
        git("clone", "--quiet", REPO_URL, str(REPO_DIR), token=token)
    git("-C", str(REPO_DIR), "fetch", "--quiet", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    git("-C", str(REPO_DIR), "checkout", "--quiet", "--force", "--detach",
        f"origin/{REPO_REF}" if is_branch else REPO_REF)
    print("commit:", git("-C", str(REPO_DIR), "rev-parse", "HEAD"))
    return REPO_DIR


def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:          # локальный запуск
        if (path / "src" / "pipeline.py").exists():
            return path
    if not IS_KAGGLE:
        raise RuntimeError("Не найден код решения (src/). Запустите ноутбук внутри репозитория.")
    root = checkout_repo()
    if not (root / "src" / "pipeline.py").exists():
        raise RuntimeError(f"В корне репозитория нет src/pipeline.py: {sorted(p.name for p in root.iterdir())}")
    return root


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

try:                                  # pymorphy3 нет в образе Kaggle
    import pymorphy3  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"], check=True)
print("repo:", REPO_ROOT, "| kaggle:", IS_KAGGLE)

commit: 23da25c9b5f853d671c332054ab3c3c05d159a78
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 67.0 MB/s eta 0:00:00
repo: /kaggle/working/avito-candgen | kaggle: True


In [2]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

from src import analysis, eda
from src.config import CFG
from src.data import load_benchmark, load_train, load_train_items_text
from src.paths import get_data_dir, get_output_dir, get_work_dir
from src.pipeline import ItemLemmaCache, add_lemma_keys, build_stage
from src.candidates import build_vocabs
from src.ranking import PoolData, coordinate_ascent, half_split_cv, predict_from_pool, weights_vector
from src.repro import library_versions, seed_everything
from src.submit import save_answer, validate_answer
from src.text import Lemmatizer
from src.utils import timer
from src.validation import (add_query_segments, build_val_queries, build_validation_split,
                            mark_seen, per_query_recall, recall_at_k)

assert CFG.n_threads == N_THREADS
seed_everything(CFG.seed)
pd.set_option("display.max_colwidth", 100)

DATA_DIR, WORK_DIR, OUT_DIR = get_data_dir(), get_work_dir(), get_output_dir()
VERSIONS = library_versions()
K, DEC = CFG.top_k, CFG.score_decimals
print(f"версия решения: {CFG.version}\ndata: {DATA_DIR}\nwork: {WORK_DIR}\nout:  {OUT_DIR}")
print(VERSIONS)

версия решения: v2
data: /kaggle/input/datasets/mm1khail/nlp-avito-interns/NLP_avito_interns/dataset
work: /kaggle/working/artifacts
out:  /kaggle/working
{'python': '3.12.13', 'numpy': '2.0.2', 'pandas': '2.3.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'pyarrow': '24.0.0', 'pymorphy3': '2.0.6', 'torch': '2.10.0+cpu'}


## 1. Данные

Все id приводятся к строкам. Описания объявлений из train не загружаются: они нужны только для позитивов валидации и дочитываются точечно (фильтр pyarrow по `item_id`).

In [3]:
with timer("загрузка"):
    train = load_train(DATA_DIR)
    bench_q, bench_items = load_benchmark(DATA_DIR)

print(f"train: {train.shape} | запросы бенчмарка: {bench_q.shape} | корпус: {bench_items.shape}")
print(f"память train ≈ {train.memory_usage(deep=True).sum() / 2**30:.2f} ГБ")
display(train.head(3))

[загрузка] 29.6 c
train: (497673, 22) | запросы бенчмарка: (2452, 10) | корпус: (189212, 14)
память train ≈ 1.63 ГБ


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,...,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_category_id,norm_text,filters_norm,query_key,rating_thr
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.0,2303374,...,54.629230,1.0,0.0,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимости за услугу Работаете с юрлица...",e8b685dffe1a408e,114,скупка телевизоров,,скупка телевизоров 652430 0 114,NaN
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.0,2303374,...,56.273392,0.0,0.0,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, жилой комплекс Новая Кузнечиха...",92e1b0446f827b59,114,автоподбор,рейтинг пользователя 4 звезды и выше,автоподбор 640860 0 рейтинг пользователя 4 звезды и выше 114,4.0
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.0,86469,...,59.783688,0.0,0.0,"Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург, садоводческое некоммерческое ...",624846856ce81d69,114,баня на дровах,"онлайн-запись тип услуги спа-услуги, массаж вид услуги красота, здоровье","баня на дровах 653240 0 онлайн-запись тип услуги спа-услуги, массаж вид услуги красота, здоровье...",NaN


## 2. EDA

«Запрос» — группа строк train с одинаковым `query_key`: нормализованный текст + локация + доставка + фильтры + категория поиска.

In [4]:
eda.query_groups(train)
OVERLAP = eda.overlap(train, bench_q, bench_items, CFG.item_stats_min_overlap)
LOCS = eda.locations(train, bench_q, bench_items)
eda.filters(train, bench_q, sample_size=50_000, seed=CFG.seed)
eda.categories(train, bench_items);


── Сколько объявлений выбирают по одному запросу ─────────────────────────
групп-запросов: 354,241 | текстов: 73,706 | объявлений: 344,825
объявлений на запрос: среднее 1.32, медиана 1, p90 2, p99 6, max 278
доли групп (5 = «5 и больше»): {1: 0.85, 2: 0.095, 3: 0.026, 4: 0.011, 5: 0.018}

── Пересечение бенчмарка с train ─────────────────────────────────────────
объявлений корпуса, встречающихся в train: 0.096
запросов, чей текст встречается в train:  0.375
запросов, чей полный ключ есть в train:   0.044
→ статистики по item_id выключены (порог 0.5)

── Локации ───────────────────────────────────────────────────────────────
пар, где локация объявления = локации поиска: 0.831
из несовпадающих: локация поиска никогда не бывает у объявлений: 0.639
запросов бенчмарка с такой («только поисковой») локацией: 0.173
расстояние до объявления при несовпадении, км: p50=16, p75=52, p90=326, p95=956
переходов «поиск → объявление»: 14,491; несовпадающих пар в переходах, встреченных ≥5 раз: 0.784
сам

**Выводы (полный прогон)**

* **Размер эталона.** У 85% запросов одно выбранное объявление, в среднем 1,32.
* **Корпус почти не пересекается с train.** Только 9,6% объявлений корпуса встречаются в train. Статистики по `item_id` (популярность, «память») автоматически выключены, искать приходится по содержанию.
* **Запросы в основном новые.** Текст знаком train у 37,5% запросов бенчмарка, полный ключ — у 4,4%. Запросы длиннохвостые, с опечатками и оборванными словами.
* **Локация — сильнейший сигнал, но не жёсткий фильтр.**
  * У 83% пар локация объявления совпадает с локацией поиска.
  * В 64% остальных пар локация поиска — «региональный» id, которого не бывает у объявлений (например, 107620 почти всегда ведёт в 637640). Таких запросов в бенчмарке 17,3%.
  * Прочие несовпадения — в основном соседние населённые пункты (медиана 16 км).
  * → признаки P(локация объявления | локация поиска) и расстояние до центра локации поиска.
* **Фильтры.** Они есть у 67% запросов train, но только у 37% бенчмарка → валидация стратифицируется по фильтрам. Фильтр по рейтингу встречается у 0,2% пар, доставка практически не встречается (признак удалён).
* **Категории.** 99,99% поисков — в категории 114, 99% корпуса — тоже. Категория поиска почти ничего не различает; различают микрокатегории.

## 3. Валидация

Псевдо-бенчмарк из train повторяет состав бенчмарка по трём осям: текст знакомый/новый × есть ли фильтр × тип локации (обычная / только поисковая). Отбор детерминирован (md5 с солью `CFG.val_salt`).

Позитивы валидации подмешиваются в **реальный корпус бенчмарка**, чтобы «чужие» объявления были ровно такими же, как в бою. Все статистики на валидации считаются только по train-фолду.

In [5]:
lem = Lemmatizer()
with timer("леммы запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)

add_query_segments(train, LOCS["item_locations"])
add_query_segments(bench_q, LOCS["item_locations"])
mark_seen(bench_q, train["norm_text"].unique())

with timer("разбиение"):
    val_keys, val_seen, fold_mask, split_report = build_validation_split(
        train, bench_q, CFG.n_val_queries, CFG.text_holdout_frac, CFG.val_salt)
    val_q, val_truth = build_val_queries(train, val_keys, val_seen)
    train_fold = train[fold_mask].reset_index(drop=True)

split_report["доля в бенчмарке"] = (split_report["цель"] / split_report["цель"].sum()).round(3)
display(split_report)
print(f"валидация: {len(val_q):,} запросов; train-фолд: {len(train_fold):,} из {len(train):,} строк")
print(f"эталонных объявлений на запрос: {np.mean([len(t) for t in val_truth]):.2f}")

[леммы запросов] 4.0 c
[разбиение] 5.9 c


,текст,страта,цель,факт,доля в бенчмарке
0,знакомый,фильтр есть | локация обычная,443,443,0.177
1,знакомый,фильтр есть | локация только поисковая,84,84,0.034
2,знакомый,фильтра нет | локация обычная,333,333,0.133
3,знакомый,фильтра нет | локация только поисковая,78,78,0.031
4,новый,фильтр есть | локация обычная,336,336,0.134
5,новый,фильтр есть | локация только поисковая,61,61,0.024
6,новый,фильтра нет | локация обычная,954,954,0.382
7,новый,фильтра нет | локация только поисковая,211,211,0.084


валидация: 2,500 запросов; train-фолд: 477,357 из 497,673 строк
эталонных объявлений на запрос: 1.22


In [6]:
# Корпус валидации = корпус бенчмарка + позитивы валидации, которых в нём нет
bench_ids = frozenset(bench_items["item_id"])            # только проверка вхождения
missing = sorted(dict.fromkeys(i for rel in val_truth for i in rel if i not in bench_ids))
extra = load_train_items_text(DATA_DIR, missing)
val_items = pd.concat([bench_items, extra[bench_items.columns]], ignore_index=True)

# Общие словари локаций и микрокатегорий — одинаковые коды на валидации и бенчмарке
vocabs = build_vocabs([train, bench_q], [train, val_items])
item_lemmas = ItemLemmaCache(lem, CFG.desc_max_chars)
print(f"корпус валидации: {len(val_items):,} (+{len(extra):,} позитивов) | "
      f"локаций: {len(vocabs.loc):,} | микрокатегорий: {len(vocabs.micro):,}")

корпус валидации: 192,080 (+2,868 позитивов) | локаций: 3,495 | микрокатегорий: 758


## 4. Пул кандидатов

Для каждого запроса объединяются списки (подробности — в `src/candidates.py`):

| Список | Отбор | K |
|---|---|---|
| `src_text` | BM25 по заголовку, параметрам, описанию + доля лемм запроса в заголовке/параметрах | 400 |
| `src_text_loc` | то же + приоритет «близких» объявлений | 400 |
| `src_prior_loc` | близкие объявления из самых вероятных микрокатегорий | 400 |
| `src_memo` | выбранные в train по такому же запросу (только при пересечении корпуса с train) | 200 |

«Близость» = max(совпадение локации, P(перехода локаций), exp(−расстояние / 30 км)).

In [7]:
val = build_stage("валидация", val_items, val_q, train_fold, lem=lem, cache=item_lemmas,
                  vocabs=vocabs, cfg=CFG, use_item_stats=OVERLAP["use_item_stats"], truth=val_truth)
n_rel = np.array([len(t) for t in val_truth], dtype=np.float64)
analysis.source_recall(val.pool, n_rel).round(4)

[валидация: индекс корпуса] 177.4 c
[валидация: статистики train] 1.1 c
  пул: 64/2500 запросов
  пул: 384/2500 запросов
  пул: 704/2500 запросов
  пул: 1024/2500 запросов
  пул: 1344/2500 запросов
  пул: 1664/2500 запросов
  пул: 1984/2500 запросов
  пул: 2304/2500 запросов
[валидация: пул кандидатов] 65.5 c
валидация: запросов 2,500, объявлений 192,080, строк пула 2,277,668 (~911 на запрос)


,recall,avg_candidates
src_text,0.6778,911.0672
src_text_loc,0.8885,911.0672
src_prior_loc,0.8578,911.0672
src_memo,0.0000,911.0672
pool,0.9644,911.0672


`recall` списка — полнота при его собственном K (сотни кандидатов); строка `pool` — потолок, выше которого никакое переранжирование пула не поднимется.

## 5. Веса формулы

Скор кандидата — взвешенная сумма признаков. Признаки, постоянные внутри запроса, на порядок кандидатов не влияют, поэтому все признаки — именно свойства пары «запрос — объявление».

Сначала несколько ориентиров, затем честная оценка подбора (учим на половине запросов, меряем на другой) и финальные веса на всей валидации.

In [8]:
data = PoolData(val.pool, val.corpus, val.features, n_rel)
W0 = weights_vector(val.features, CFG.init_weights)
references = {
    "только текст": weights_vector(val.features, {"title": 1, "params": 0.5, "desc": 0.3, "cov": 1}),
    "текст + та же локация": weights_vector(val.features, {"title": 1, "params": 0.5, "desc": 0.3,
                                                           "cov": 1, "loc_same": 1}),
    "веса v1 (старт подбора)": W0,
}
for name, w in references.items():
    print(f"{name:28s} recall@{K} = {data.recall(w, K, DEC):.4f}")

только текст                 recall@50 = 0.3233
текст + та же локация        recall@50 = 0.7089
веса v1 (старт подбора)      recall@50 = 0.8159


In [9]:
with timer("оценка подбора на половинах"):
    CV = half_split_cv(data, W0, CFG.tune_grid, CFG.tune_passes, K, DEC)
for name, value in CV.items():
    print(f"{name}: recall@{K} на отложенной половине = {value:.4f}")
print(f"среднее: {np.mean(list(CV.values())):.4f}")

[оценка подбора на половинах] 133.0 c
чётные→нечётные: recall@50 на отложенной половине = 0.8736
нечётные→чётные: recall@50 на отложенной половине = 0.8807
среднее: 0.8772


In [10]:
with timer("подбор весов"):
    W, _ = coordinate_ascent(data, W0, CFG.tune_grid, CFG.tune_passes, K, DEC)
WEIGHTS = dict(zip(val.features, map(float, W)))
pd.Series(WEIGHTS, name="вес").to_frame().T

  старт: recall@50 = 0.81587
  проход 1: recall@50 = 0.85299 | title=1, desc=0.5, cov=0.5, filt=1, filt_exact=0.1, logp=0.1, loc_same=2, loc_p=2, loc_logp=0.1
  проход 2: recall@50 = 0.87860 | title=2, params=0.1, desc=2, cov=1, filt=2, filt_exact=0.25, logp=0.1, loc_same=2, loc_p=2, loc_logp=0.25
[подбор весов] 66.0 c


,title,params,desc,cov,filt,filt_exact,logp,loc_same,loc_p,loc_logp,log_dist,rating_ok,rating,log_reviews
вес,2.0,0.1,2.0,1.0,2.0,0.25,0.1,2.0,2.0,0.25,0.0,0.0,0.0,0.0


## 6. Анализ качества

Метрика пересчитывается «как на платформе» — по спискам `item_id` — и сверяется с быстрой оценкой по пулу.

In [11]:
val_pred = predict_from_pool(val.pool, val.corpus, val.features, W, K, DEC, len(val_q))
VAL_RECALL = recall_at_k(val_pred, val_truth, K)
assert abs(VAL_RECALL - data.recall(W, K, DEC)) < 1e-9, "метрики по пулу и по спискам разошлись"
print(f"Recall@{K} на валидации: {VAL_RECALL:.4f} (веса подобраны на ней же); "
      f"честная оценка: {np.mean(list(CV.values())):.4f}")

r50 = per_query_recall(val_pred, val_truth, K)
pool_hit = analysis.pool_hit_rate(val.pool, n_rel)
analysis.segment_table(val_q, r50, pool_hit)

Recall@50 на валидации: 0.8786 (веса подобраны на ней же); честная оценка: 0.8772


n   share    pool  recall50
ось        сегмент                                                 
seg_text   знакомый                   938  0.3752  0.9752    0.9049
           новый                     1562  0.6248  0.9578    0.8628
seg_filter фильтр есть                924  0.3696  0.9742    0.9087
           фильтра нет               1576  0.6304  0.9586    0.8609
seg_loc    локация обычная           2066  0.8264  0.9828    0.9119
           локация только поисковая   434  0.1736  0.8764    0.7200

In [12]:
errors = analysis.error_examples(val_pred, val_truth, val_q, val_items, pool_hit, n=20)
print(f"запросов без единого попадания: {int((r50 == 0).sum())} ({(r50 == 0).mean():.3f}); "
      f"эталон был в пуле у {np.mean(pool_hit[r50 == 0] > 0):.3f} из них")
errors

запросов без единого попадания: 284 (0.114); эталон был в пуле у 0.704 из них


,запрос,фильтры,та же локация,в пуле,заголовок эталона,параметры эталона
0,покос травы,"Вид услуги Сад, благоустройство",False,True,Комплексные услуги Садовника Универсала,"Вид услуги Сад, благоустройство Место оказания услуг Дачная ул. Тип стоимости за услугу Начальна..."
1,аренда зала,"Вид услуги Праздники, мероприятия",False,False,"Бассейн, Беседка, закрытая терраса. День рождения","Вид услуги Праздники, мероприятия Место оказания услуг Ростов-на-Дону, улица Малое Зелёное Кольц..."
2,шиномонтаж,"Тип услуги автосервиса Шиномонтаж и ремонт дисков Вид услуги Автосервис, аренда Тип услуги Автос...",False,True,Шиномонтаж для таксопарка,"Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Москва, улица Борисовск..."
3,манипулятор одинцово,,False,True,"Аренда и услуги манипулятора, стрела 3 т, 14.5 м, борт 5.6 т, 4.7 м","Вид услуги Автосервис, аренда Тип услуги Аренда спецтехники Место оказания услуг ул. Маршала Жук..."
4,аренда дозиметра,,True,False,Измерение радиации мкс-01са1М,"Вид услуги Охрана, безопасность Место оказания услуг Санкт-Петербург, Ланское шоссе, 24к3 Тип ст..."
5,прокат квадроциклов,"Вид услуги Автосервис, аренда",False,True,"Квадроциклы, Мотоциклы, Баня в подарок, Шашлыки","Вид услуги Автосервис, аренда Тип услуги Аренда авто Место оказания услуг Ленинградская область,..."
6,укладка брусчатки,"Вид услуги Сад, благоустройство",False,True,Укладка брусчатки благоустройство территория,"Вид услуги Сад, благоустройство Место оказания услуг Московская область, Лобня, Красноармейская ..."
7,монтажник,,True,False,"Установка видеонаблюдения, скуд, домофона монтаж","Вид услуги Охрана, безопасность Место оказания услуг Москва, улица Земляной Вал, 24/30с1 Тип сто..."
8,работы по дереву,Вид услуги Строительство,False,False,"Замена венцов, поднятие домов","Вид услуги Строительство Место оказания услуг Московская обл., Подольск, Заводская ул. Работа по..."
9,нейровидео,,True,True,Создание ии мультфильмов в стиле Disney,"Вид услуги Деловые услуги Тип услуги Место оказания услуг Ростовская обл., Шахты Опыт работы 10 ..."


## 7. Предсказание для бенчмарка

Статистики пересчитываются по **всему** train, веса берутся из раздела 5.

In [13]:
bench = build_stage("бенчмарк", bench_items, bench_q, train, lem=lem, cache=item_lemmas,
                    vocabs=vocabs, cfg=CFG, use_item_stats=OVERLAP["use_item_stats"])
assert bench.features == val.features
bench_pred = predict_from_pool(bench.pool, bench.corpus, bench.features, W, K, DEC, len(bench_q))

ANSWER_PATH = OUT_DIR / "answer.csv"
save_answer(bench_q["query_id"], bench_pred, ANSWER_PATH)
CHECK = validate_answer(ANSWER_PATH, bench_q["query_id"], bench_items["item_id"], K)
print(CHECK)
pd.read_csv(ANSWER_PATH, dtype=str, nrows=3)

[бенчмарк: индекс корпуса] 50.3 c
[бенчмарк: статистики train] 1.2 c
  пул: 64/2452 запросов
  пул: 384/2452 запросов
  пул: 704/2452 запросов
  пул: 1024/2452 запросов
  пул: 1344/2452 запросов
  пул: 1664/2452 запросов
  пул: 1984/2452 запросов
  пул: 2304/2452 запросов
[бенчмарк: пул кандидатов] 62.1 c
бенчмарк: запросов 2,452, объявлений 189,212, строк пула 2,358,226 (~962 на запрос)
{'rows': 2452, 'min_items': 50, 'max_items': 50, 'md5': '2de61da58afd87fc244e225b7cccde58'}


,query_id,answer
0,70DfDUpwjxB4lzFd,d722bcda1a555091 355392014208b7bf 4494aa11cbdc2d65 255fbeaf526a1cc1 b92ee8f432cec2d1 603623b4bd9...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 1c6d2079e07f4497 367af128a9ea2a48 431efbc80d59155f 14fbc6d1a2f...
2,LZCZNoVG4AFUkVRJ,168a9207e80b0be4 3b370cc603f67947 3f89b8062dc85f1c dab52187b4500d9b d8fce513e4f000a7 edecb3695ab...


## 8. Артефакты

Отчёт с метриками, весами, конфигом и версиями библиотек; пулы с признаками пригодятся для обучения ранкера. Повторный запуск должен дать тот же md5 `answer.csv`.

In [14]:
report = {
    "version": CFG.version,
    "answer_md5": CHECK["md5"],
    "val_recall@50": VAL_RECALL,
    "val_recall@50_half_cv": CV,
    "val_pool_recall": analysis.source_recall(val.pool, n_rel)["recall"].to_dict(),
    "weights": WEIGHTS,
    "eda": {k: v for k, v in OVERLAP.items()},
    "split": split_report.to_dict(orient="records"),
    "config": CFG.as_dict(),
    "versions": VERSIONS,
}
(WORK_DIR / f"report_{CFG.version}.json").write_text(json.dumps(report, ensure_ascii=False, indent=1))
val.pool.to_parquet(WORK_DIR / f"val_pool_{CFG.version}.parquet", index=False)
bench.pool.to_parquet(WORK_DIR / f"bench_pool_{CFG.version}.parquet", index=False)
val_q.assign(truth=[" ".join(t) for t in val_truth]).to_parquet(
    WORK_DIR / f"val_queries_{CFG.version}.parquet", index=False)

print(f"Recall@50 валидация: {VAL_RECALL:.4f} | честная оценка: {np.mean(list(CV.values())):.4f}")
print(f"answer.csv md5: {CHECK['md5']}")

Recall@50 валидация: 0.8786 | честная оценка: 0.8772
answer.csv md5: 2de61da58afd87fc244e225b7cccde58
